In [72]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

In [73]:
# 브라우저 열기
browser = webdriver.Chrome()

In [74]:
# 스타벅스 지역별 매장검색 페이지에 접속
url = 'https://www.starbucks.co.kr/store/store_map.do?disp=locale'
browser.get(url)

In [75]:
# 시/도 선택 --> 전체

# 서울 선택 --> 전체 선택

In [76]:
# 서울 -> 전체 매장정보 수집하기
soup = BeautifulSoup(browser.page_source, 'html.parser')

# 매장정보 뭉치(html) 찾기
shop_part_list = soup.select('ul.quickSearchResultBoxSidoGugun > li.quickResultLstCon')
len(shop_part_list)

0

In [77]:
# 1개 매장 정보 선택 --> 필요한 정보 추출/정리
shop_part = shop_part_list[1]
shop = shop_part.select('strong')[0].text.strip()
address = shop_part.select('p')[0].text.replace('1522-3232', '')
lat = shop_part['data-lat']
long = shop_part['data-long']
# shop_type = shop_part.select('i')[0]['class'][0].replace('pin_', '')
shop_type = shop_part.select('i')[0]['class'][0][4:]
shop_type
# lat: 위도(남북)  long: 경도(동서)

IndexError: list index out of range

In [ ]:
for shop_part in shop_part_list:
    shop = shop_part.select('strong')[0].text.strip()
    address = shop_part.select('p')[0].text.replace('1522-3232', '')
    lat = shop_part['data-lat']
    long = shop_part['data-long']
    shop_type = shop_part.select('i')[0]['class'][0][4:]
    print(shop, address, lat, long, shop_type)

In [ ]:
# 스타벅스 홈페이지 접속

# 서울 --> 전체 클릭

# 현재 브라우저에 나오는 모든 매장 정보 수집하기
def get_sido_starbucks(browser):
    soup = BeautifulSoup(browser.page_source, 'html.parser')
    shop_part_list = soup.select('ul.quickSearchResultBoxSidoGugun > li.quickResultLstCon')

    results = []
    for shop_part in shop_part_list:
        shop = shop_part.select('strong')[0].text.strip()
        address = shop_part.select('p')[0].text.replace('1522-3232', '')
        lat = shop_part['data-lat']
        long = shop_part['data-long']
        shop_type = shop_part.select('i')[0]['class'][0][4:]
        data = [shop, address, lat, long, shop_type]
        results.append(data)

    return results

In [ ]:
results = get_sido_starbucks(browser)
results

In [79]:
# iferror(실행할수식, 에러발생시실행할수식)
# try:
#     실행코드
# except:
#     에러발생시실행할코드

In [ ]:
# 시/도 선택 목록 찾기
region_list = browser.find_elements('css selector', 'a.set_sido_cd_btn')

# 시/도 선택
region = region_list[0]
region.click()
time.sleep(0.5)

# <전체> 선택
select_all = browser.find_elements('css selector', 'a.set_gugun_cd_btn')[0]
select_all.click()
time.sleep(3)

# 전체 매장 정보 --> 수집하기
sido_starbucks_list = get_sido_starbucks(browser)

# 지역 검색 클릭하기
search_sido_btn = browser.find_elements('css selector', 'h3.on > a')[0]
search_sido_btn.click()
time.sleep(0.5)

sido_starbucks_list

In [83]:
! pip install tqdm

In [84]:
from tqdm.notebook import tqdm

for i in tqdm(range(10)):
    print(i)
    time.sleep(0.5)

  0%|          | 0/10 [00:00<?, ?it/s]

0
1
2
3
4
5
6
7
8
9


In [87]:
# 시/도 선택 목록 찾기
region_list = browser.find_elements('css selector', 'a.set_sido_cd_btn')

results = []
# 시/도 선택
#region = region_list[0]
for region in tqdm(region_list):
    region.click()
    time.sleep(0.5)
    
    # <전체> 선택
    try:
        select_all = browser.find_elements('css selector', 'a.set_gugun_cd_btn')[0]
        select_all.click()
        time.sleep(3)
    except:
        print('전체 선택이 안돼서 넘어감')
    
    # 전체 매장 정보 --> 수집하기
    sido_starbucks_list = get_sido_starbucks(browser)
    results = results + sido_starbucks_list
    
    # 지역 검색 클릭하기
    search_sido_btn = browser.find_elements('css selector', 'h3.on > a')[0]
    search_sido_btn.click()
    time.sleep(0.5)
    
    sido_starbucks_list

  0%|          | 0/17 [00:00<?, ?it/s]

전체 선택이 안돼서 넘어감


In [88]:
len(results)

2118

In [93]:
import pandas as pd
df = pd.DataFrame(results)
df.columns = ['매장명', '주소', '위도', '경도', '매장타입']
df.to_excel('./starbucks_list.xlsx', index=False)

In [94]:
raw = pd.read_excel('./starbucks_list.xlsx')

In [95]:
raw

,매장명,주소,위도,경도,매장타입
0,역삼아레나빌딩,서울특별시 강남구 언주로 425 (역삼동),37.501087,127.043069,general
1,논현역사거리,서울특별시 강남구 강남대로 538 (논현동),37.510178,127.022223,general
2,신사역성일빌딩,서울특별시 강남구 강남대로 584 (논현동),37.513931,127.020606,general
3,국기원사거리,서울특별시 강남구 테헤란로 125 (역삼동),37.499517,127.031495,general
4,대치재경빌딩,서울특별시 강남구 남부순환로 2947 (대치동),37.494668,127.062583,general
...,...,...,...,...,...
2113,세종해밀DT,세종특별자치시 미리내로 171 (해밀동),36.523513,127.267640,generalDT
2114,세종소담,"세종특별자치시 소담3로 8 (소담동) 1동 12호,13호,14호,15호,15a호,16호",36.485755,127.300509,general
2115,세종예술의전당,세종특별자치시 나성동로 6 (나성동),36.486552,127.269712,general
2116,세종금강DT,세종특별자치시 시청대로 45 (대평동),36.476054,127.277498,generalDT


## 지도 시각화

In [96]:
! pip install folium


   -------------------- ------------------- 1/2 [folium]
   ---------------------------------------- 2/2 [folium]



In [123]:
import folium

In [117]:
# 지도 만들기
# url에서 !3d 뒤: 위도, !4d 뒤: 경도
seoul_station = [37.555474, 126.972616]
m = folium.Map(location = seoul_station, zoom_start = 12)
m


## 스타벅스 지도 만들기

In [115]:
# 지도 마커 추가하기
# folium.Marker(seoul_station,popup = 'seoul', tooltip = '서울역').add_to(m)
# m

In [118]:
# 스타벅스 매장정보 추가

for i in raw.index:
    lat = raw.loc[i, '위도']
    long = raw.loc[i, '경도']
    name = raw.loc[i, '매장명']
    shop_type = raw.loc[i, '매장타입']
    address = raw.loc[i, '주소']

    folium.Marker(lat, long, tooltip = name, popup = address).add_to(m)

m

TypeError: Marker.__init__() got multiple values for argument 'popup'

## 스타벅스 지도 만들기.클러스터

In [124]:
from folium.plugins import MarkerCluster

In [127]:
# 지도 만들기
seoul_station = [37.5558824, 126.9715573]
m = folium.Map(location = seoul_station, 
          zoom_start = 12)

# 클러스터 생성 --> 지도에 추가하기
marker_cluster = MarkerCluster().add_to(m)

# 스타벅스 매장정보 추가
for i in raw.index:
    lat = raw.loc[i, '위도']
    long = raw.loc[i, '경도']
    name = raw.loc[i, '매장명']
    shop_type = raw.loc[i, '매장타입']
    address = raw.loc[i, '주소']
    
    folium.Marker( [lat, long ], tooltip = name, popup = address).add_to(marker_cluster)

# 미니맵 만들어서 지도에 추가
minimap = MiniMap()
minimap.add_to(m)

NameError: name 'MiniMap' is not defined